# Iris Flower Classification — Exploratory Data Analysis

Load the Iris dataset, preprocess, perform EDA, train a Random Forest, and evaluate the model.

In [ ]:
%matplotlib inline

import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import joblib

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
SRC_DIR = PROJECT_ROOT / "src"
sys.path.insert(0, str(SRC_DIR))

from data_preprocessing import (
    load_dataset,
    check_missing_values,
    display_dataset_info,
    encode_target_labels,
    prepare_features_and_target,
    split_train_test,
    TARGET_COLUMN,
)

DATA_PATH = PROJECT_ROOT / "data" / "Iris.csv"
MODEL_PATH = PROJECT_ROOT / "models" / "iris_model.pkl"

## 1. Load Dataset

In [ ]:
df = load_dataset(DATA_PATH)
print("Shape:", df.shape)
df.head()

## 2. Data Preprocessing

In [ ]:
print("Missing values:")
print(check_missing_values(df))
display_dataset_info(df)

In [ ]:
df_encoded, label_encoder = encode_target_labels(df)
X, y = prepare_features_and_target(df_encoded)
X_train, X_test, y_train, y_test = split_train_test(X, y, test_size=0.2, random_state=42)
print(f"Train: {len(X_train)}, Test: {len(X_test)}")

## 3. Exploratory Data Analysis (EDA)

In [ ]:
print("First 5 rows:")
display(df.head())

print("\nClass distribution:")
print(df[TARGET_COLUMN].value_counts())

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
df[TARGET_COLUMN].value_counts().plot(kind="bar", ax=ax, color=["#5B9BD5", "#ED7D31", "#70AD47"])
ax.set_title("Class Distribution")
ax.set_ylabel("Count")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
feature_cols = [c for c in df.columns if c not in ("Id", TARGET_COLUMN)]
sns.pairplot(df, vars=feature_cols, hue=TARGET_COLUMN, palette="Set2")
plt.suptitle("Feature Pairplot", y=1.02)
plt.show()

In [ ]:
plt.figure(figsize=(8, 6))
sns.heatmap(df[feature_cols].corr(), annot=True, cmap="coolwarm", center=0, fmt=".2f")
plt.title("Correlation Heatmap")
plt.tight_layout()
plt.show()

## 4. Train Random Forest Classifier

In [ ]:
model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train, y_train)
y_pred = model.predict(X_test)

accuracy = accuracy_score(y_test, y_pred)
print(f"Model Accuracy: {accuracy * 100:.2f}%")

## 5. Model Evaluation

In [ ]:
cm = confusion_matrix(y_test, y_pred)
print("Confusion Matrix:")
print(cm)
print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=label_encoder.classes_))

In [ ]:
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=label_encoder.classes_,
            yticklabels=label_encoder.classes_)
plt.title("Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.tight_layout()
plt.show()

## 6. Save Model

In [ ]:
MODEL_PATH.parent.mkdir(parents=True, exist_ok=True)
joblib.dump(model, MODEL_PATH)
print(f"Model saved to: {MODEL_PATH}")